### Work in your own copy
Save a working copy of this notebook before editing.

# **OTH Course Knowledge Representation 2026**
## **Tutorial 3: Labelled Property Graphs with Memgraph**

Original tutorial: Sara Buchmann, sara.buchmann@oth-regensburg.de.  
Revised setup, task wording, and reference answers: 21 November 2026.

We create and query labelled property graphs, translate a small CSV dataset into a graph, and explore a larger comic dataset. **The database is Memgraph; the Neo4j Python package is only the Bolt client.**


## Before you start: install and start Memgraph
<a id="memgraph-setup"></a>

**Prepare this setup before class.** Use the supplied `compose.yaml` in this tutorial folder. It starts the Memgraph database (with MAGE) and Memgraph Lab together. The course versions are **Memgraph MAGE 3.14.0** and **Lab 3.14.2**; both were used for the complete tutorial test.

Official instructions: [Memgraph installation](https://memgraph.com/docs/getting-started/install-memgraph) and [Memgraph with Docker Compose](https://memgraph.com/docs/getting-started/install-memgraph/docker-compose).

### 1. Install and start Docker

| Operating system | One-time preparation |
|---|---|
| macOS | Install [Docker Desktop for Mac](https://docs.docker.com/desktop/setup/install/mac-install/), choosing **Apple Silicon** or **Intel** to match your processor. Launch the application and wait until Docker is running. |
| Windows | Install [Docker Desktop for Windows](https://docs.docker.com/desktop/setup/install/windows-install/), complete its WSL 2 setup, and use **Linux containers**. Launch Docker Desktop. |
| Linux | Install [Docker Engine](https://docs.docker.com/engine/install/) and the [Docker Compose plugin](https://docs.docker.com/compose/install/linux/). Start the Docker service as described for your distribution. |

Docker Desktop already includes Compose. If you already use [OrbStack on macOS](https://docs.orbstack.dev/docker/), start OrbStack and use its Docker/Compose commands for the following steps.

### 2. Start the tutorial services

Run the following commands in a **terminal**, in the folder containing this notebook and `compose.yaml`. From the `kg-oth` repository root, first change into that folder:

```bash
cd 03-tutorials/tut3/LPG_Memgraph
```

Check that Docker is running, then start the main database:

```bash
docker info
docker compose version
docker compose up -d --wait
docker compose ps
```

The first start downloads the images and needs an internet connection. `up -d --wait` leaves the services running in the background and waits for readiness. In `docker compose ps`, the `memgraph` service should be **healthy** and `lab` should be running.

For **Part 3**, also start the separate Marvel stack:

```bash
docker compose --env-file .env.marvel -p tut03-marvel up -d --wait
docker compose --env-file .env.marvel -p tut03-marvel ps
```

This creates a second database with separate storage and ports. Before class, follow the [Marvel preparation instructions](#marvel-graph) to load its dataset **once** into the empty Marvel instance. Starting the containers alone does not load that dataset.

### 3. Open Lab and check the connection

| Use | Main stack: Parts 1–2 | Marvel stack: Part 3 |
|---|---|---|
| Browser: Memgraph Lab | [http://localhost:3000](http://localhost:3000) | [http://localhost:3001](http://localhost:3001) |
| Local Python: database URI | `bolt://localhost:7687` | `bolt://localhost:7690` |
| Lab's connection inside its Compose network | host `memgraph`, port `7687` | host `memgraph`, port `7687` |

Lab is configured to connect automatically. If entering a connection manually in either containerized Lab, use **host `memgraph`**, **port `7687`**, empty username/password, and **no TLS** for this local course setup. Each Lab connects to the database in its own stack.

Open the query editor in Lab and run:

```cypher
RETURN 1 AS ready;
```

A result with `ready = 1` confirms that queries reach the database. Next, set up the Python environment below; the notebook's connection cell performs the same check from Python.

### 4. Resume, stop, or diagnose the setup

Run `docker compose up -d --wait` again when returning to the main exercises; use the Marvel command above for Part 3. The named volumes retain imported data. To pause both stacks:

```bash
docker compose stop
docker compose --env-file .env.marvel -p tut03-marvel stop
```

Keep the named volumes when stopping or removing containers; `docker compose down -v` would delete the stored exercise data.

- **Cannot connect to the Docker daemon:** start Docker Desktop or OrbStack, then check `docker info` again.
- **`docker compose` is unavailable:** follow the [Compose installation guide](https://docs.docker.com/compose/install/).
- **A service does not become ready:** inspect its messages with `docker compose logs --tail=50 memgraph lab`. For the Marvel stack, add `--env-file .env.marvel -p tut03-marvel` before `logs`.

Documentation and course setup checked on **21 November 2026**.


### Python environment

Use Python **3.11 or newer**; Python **3.12** is the recommended course interpreter. Create a new environment rather than reusing a copied environment from another computer.

**macOS / Linux**

```bash
python3 -m venv .venv-tut03
source .venv-tut03/bin/activate
python -m pip install -r requirements.txt
python -m jupyter lab
```

**Windows PowerShell**

```powershell
py -m venv .venv-tut03
.\.venv-tut03\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m jupyter lab
```

Select the kernel belonging to that environment. The requirements list only direct dependencies. **GQLAlchemy and pymgclient are not required** for this tutorial.

Memgraph documents the [Neo4j Python driver as a supported client](https://memgraph.com/docs/client-libraries/python). No Neo4j database is installed by installing this package.


### How to use this notebook

- Each **Task** is immediately followed by its own Python code cell. Write Cypher inside a triple-quoted Python string and call `run_cypher(query)`. Raw Cypher is not Python.
- Expand **Show solution** after trying the task. The closed Markdown reference does not execute automatically; the model-solution edition puts that code into the working cell.
- Run the preparation cells in order. Section 1 creates its graph in task **1.0**; Section 2 has a supplied importer. Section 3 explicitly switches the connection to a separate Marvel instance.
- When returning from Marvel to Part 1, rerun the initial connection cell. For Part 2, rerun its connection/import cell. A connection continues to point to its last selected instance until changed.
- Code examples in the introductory explanations are illustrations. They are not extra numbered tasks.
- This tutorial uses a **local Jupyter kernel** and a local Memgraph server. In Colab, `localhost` points to the Colab machine, not your laptop. Use a local kernel or an explicitly configured reachable remote Memgraph instance.
- Keep the notebook with `tutorial_runtime.py`, `tutorial_data.py`, `compose.yaml`, `.env.marvel`, `requirements.txt`, and the small `data/` directory. 
- LIMIT examples include an ordering rule. Lists and distinct entities are compared by meaning; row order is not otherwise guaranteed.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "LPG_Memgraph",
    Path.cwd() / "03-tutorials" / "tut3" / "LPG_Memgraph",
    Path.cwd() / "kg-oth" / "03-tutorials" / "tut3" / "LPG_Memgraph",
]
NOTEBOOK_DIR = next(
    (path.resolve() for path in candidates
     if (path / "tutorial_data.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Open Jupyter in the LPG_Memgraph tutorial folder.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
print("Tutorial folder:", NOTEBOOK_DIR)


If packages were installed before class, skip the following cell. If using an existing notebook kernel, `%pip` installs into that kernel; restart it if Jupyter requests a restart.

In [ ]:
%pip install -r requirements.txt


In [ ]:
from IPython.display import display
import pandas as pd
from tutorial_runtime import connect, run_cypher, load_flights, close
from tutorial_data import STUDENTS, SOCIAL_RELATIONSHIPS, get_flight_data

print("Connected to:", connect())
display(run_cypher("RETURN 1 AS ready"))


## Cypher: a short bridge from SPARQL

| Purpose | Cypher |
|---|---|
| Node and label pattern | `(s:Student)` |
| Directed relationship and variable | `(s)-[r:KNOWS]->(t)` |
| Property equality in a pattern | `(s:Student {name: "Anna"})` |
| Extra conditions | `WHERE s.age > 24` |
| Projection and duplicate removal | `RETURN DISTINCT s.name` |
| Grouping and a later filter | `WITH s, count(DISTINCT t) AS n WHERE n > 1` |
| Missing matches | `OPTIONAL MATCH`; use `IS NULL` for missing values |
| Bounded variable length | `[:BEST_FRIEND*2]` means exactly two relationships |

The helper returns a **pandas DataFrame**, including column names for empty results:

```python
query = """
MATCH (s:Student)
WHERE s.age > 24
RETURN s.name AS student, s.age AS age
ORDER BY student
"""
display(run_cypher(query))
```

Nodes and relationships have identity. Within one `MATCH`, a relationship cannot be reused for another relationship pattern. Different node variables can still denote the same node unless the pattern or a condition excludes that.

Use parameters for data values. The supplied imports use `MERGE` on stable fixture keys, so rerunning the unchanged import does not create a second copy. Avoid running a whole-database `DETACH DELETE` when you only mean to reload one exercise.

In Lab, paste just the Cypher string, not the surrounding Python. The normal graph view and Query execution are alternative ways to inspect the same Memgraph instance.

### Creating graph data
`CREATE` adds new elements; rerunning it can create duplicates. `MERGE` looks for a whole specified pattern and creates it when absent. For controlled imports, merge nodes by their fixture key before merging the relationships.

Illustrative Cypher, separate from the student fixture:

```cypher
MERGE (a:Person {name: "Example A"})
MERGE (b:Person {name: "Example B"})
MERGE (a)-[:KNOWS]->(b)
RETURN a, b;
```

A value such as `r.since` is a relationship property. A connection to another entity is represented by a relationship, not by storing a node reference as an ordinary property value.


# Part 1: Student Social Network
<a id="student-network"></a>

Model these **ten fictional students** as `Student` nodes with `name`, `age`, and `field_of_study`.

| Name | Age | Field of study |
|---|---:|---|
| Anna | 22 | Data Science |
| Ben | 24 | Computer Science |
| Clara | 21 | Medicine |
| David | 25 | Mathematics |
| Eva | 23 | Design |
| Frank | 26 | Economics |
| Grace | 22 | Physics |
| Hannah | 20 | Computer Science |
| Ian | 24 | Mathematics |
| Julia | 21 | Data Science |

Use the following relationships:

- Anna is best friends with Eva, knows Ben, and loves David.
- Ben and David are best friends. Ben loves Anna, knows Eva, and hates Clara.
- David knows Clara and loves Julia.
- Clara is best friends with Julia and knows Anna.
- Eva loves Ben and knows Grace and Frank.
- Frank is best friends with Grace and hates Ben.
- Grace loves Ben and is best friends with Clara.
- Hannah knows Clara and is best friends with Julia.
- Ian and Frank are best friends. Ian loves Julia.
- Julia is best friends with Eva and hates Anna.

**Conventions:** `KNOWS`, `LOVES`, and `HATES` are directed. Store each `BEST_FRIEND` pair as **two opposite directed relationships**, and use directed friendship patterns in the tasks. The resulting graph has **10 nodes and 32 relationships**.

For practising **relationship properties**, the supplied `KNOWS` data additionally contain fictional `since` years. These are illustrative teaching annotations, not facts inferred from the original description. The complete input records are available as `STUDENTS` and `SOCIAL_RELATIONSHIPS`.


## 1.0) Create the knowledge graph
<a id="exercise-1-0"></a>

**Task:** Create all students and the four relationship types from the supplied records. Use `MERGE` with the unique student name as the fixture key, pass values as parameters, and preserve the optional `since` relationship property. Rerunning your code should leave **10 students and 32 relationships**, not duplicate the fixture.

**Hint:** first create the students. Then group relationships by the four fixed type names; a relationship type is syntax, while names and years are parameter values. If setup blocks the later tasks, study this reference import and continue with the queries.


In [ ]:
# Exercise 1.0 -- add your code here


<details>
<summary>Show solution — Exercise 1.0</summary>

```python
run_cypher("""
UNWIND $rows AS row
MERGE (s:Student {name: row.name})
SET s.age = row.age, s.field_of_study = row.field_of_study
""", {"rows": STUDENTS})

for relationship_type in ("BEST_FRIEND", "KNOWS", "LOVES", "HATES"):
    rows = [
        {"source": row["source"], "target": row["target"],
         "since": row.get("since")}
        for row in SOCIAL_RELATIONSHIPS
        if row["type"] == relationship_type
    ]
    # Only these four fixed type names enter the query text.
    run_cypher(f"""
UNWIND $rows AS row
MATCH (a:Student {{name: row.source}}),
      (b:Student {{name: row.target}})
MERGE (a)-[r:{relationship_type}]->(b)
SET r.since = row.since
""", {"rows": rows})

display(run_cypher("MATCH (s:Student) RETURN count(s) AS students"))
answer_1_0 = run_cypher("""
MATCH (:Student)-[r]->(:Student)
RETURN type(r) AS relationship, count(r) AS relationships
ORDER BY relationship
""")
display(answer_1_0)
```

**Checkpoint:** 10 students; `BEST_FRIEND = 16`, `HATES = 3`, `KNOWS = 7`, `LOVES = 6`. The eight friendship pairs explain the sixteen friendship relationships. `MERGE` makes the unchanged fixture repeatable; it does not clean up duplicates from an earlier `CREATE` import.

</details>


## 1.1) Students older than 24
<a id="exercise-1-1"></a>

**Task:** Return each student's name and age where age is strictly greater than 24. Sort by name.


In [ ]:
# Exercise 1.1 -- add your code here


<details>
<summary>Show solution — Exercise 1.1</summary>

```python
answer_1_1 = run_cypher("""
MATCH (s:Student)
WHERE s.age > 24
RETURN s.name AS student, s.age AS age
ORDER BY student
""")
display(answer_1_1)
```

**Expected:** David (25), Frank (26).

</details>


## 1.2) Computer Science students 
<a id="exercise-1-2"></a>

**Task:** Return the names of all students studying Computer Science, sorted by name.


In [ ]:
# Exercise 1.2 -- add your code here


<details>
<summary>Show solution — Exercise 1.2</summary>

```python
answer_1_2 = run_cypher("""
MATCH (s:Student {field_of_study: "Computer Science"})
RETURN s.name AS student
ORDER BY student
""")
display(answer_1_2)
```

**Expected:** Ben, Hannah.

</details>


## 1.3) Eva's outgoing connections and relationship properties
<a id="exercise-1-3"></a>

**Task:** Return the students Eva **knows**, together with the `since` property of each `KNOWS` relationship. Do not substitute `BEST_FRIEND` for `KNOWS`. Sort by student name.


In [ ]:
# Exercise 1.3 -- add your code here


<details>
<summary>Show solution — Exercise 1.3</summary>

```python
answer_1_3 = run_cypher("""
MATCH (:Student {name: "Eva"})-[r:KNOWS]->(s:Student)
RETURN s.name AS student, r.since AS since
ORDER BY student
""")
display(answer_1_3)
```

**Expected:** Frank (2023), Grace (2025). The year is stored on the relationship, not on either student.

</details>


## 1.4) Students in David's field 
<a id="exercise-1-4"></a>

**Task:** Return the students studying the same field as David, excluding David himself. Compare node identity rather than assuming different variables mean different students.


In [ ]:
# Exercise 1.4 -- add your code here


<details>
<summary>Show solution — Exercise 1.4</summary>

```python
answer_1_4 = run_cypher("""
MATCH (david:Student {name: "David"}), (s:Student)
WHERE s.field_of_study = david.field_of_study AND s <> david
RETURN s.name AS student
ORDER BY student
""")
display(answer_1_4)
```

**Expected:** Ian.

</details>


## 1.5) Friends of Anna's friends
<a id="exercise-1-5"></a>

**Task:** Follow **exactly two** `BEST_FRIEND` relationships from Anna. Exclude Anna herself and return distinct student names. Here “friend” means `BEST_FRIEND` only.


In [ ]:
# Exercise 1.5 -- add your code here


<details>
<summary>Show solution — Exercise 1.5</summary>

```python
answer_1_5 = run_cypher("""
MATCH (anna:Student {name: "Anna"})-[:BEST_FRIEND*2]->(s:Student)
WHERE s <> anna
RETURN DISTINCT s.name AS student
ORDER BY student
""")
display(answer_1_5)
```

**Expected:** Julia. With reciprocal stored edges, Anna could also be reached through two different opposite edges, so the explicit exclusion matters.

</details>


## 1.6) Closed love triangles 
<a id="exercise-1-6"></a>

**Task:** Find all student names participating in a **directed cycle of length three**: A loves B, B loves C, and C loves A. Require three different students. An open chain of two `LOVES` relationships is not a triangle. Explain an empty result.


In [ ]:
# Exercise 1.6 -- add your code here


<details>
<summary>Show solution — Exercise 1.6</summary>

```python
answer_1_6 = run_cypher("""
MATCH (a:Student)-[:LOVES]->(b:Student)
      -[:LOVES]->(c:Student)-[:LOVES]->(a)
WHERE a <> b AND b <> c AND c <> a
RETURN DISTINCT a.name AS student
ORDER BY student
""")
display(answer_1_6)
```

**Expected:** no rows. The original fixture has open love chains but no closed three-cycle. As a reasoning exercise, adding Julia → Anna would close Anna → David → Julia → Anna; keep the base graph unchanged for the remaining tasks.

</details>


## 1.7) Knowing someone who hates another student (required homework)
<a id="exercise-1-7"></a>

**Task:** Return each matching triple: the student, the person they know, and the person that known person hates. Sort by the three names.


In [ ]:
# Exercise 1.7 -- add your code here


<details>
<summary>Show solution — Exercise 1.7</summary>

```python
answer_1_7 = run_cypher("""
MATCH (s:Student)-[:KNOWS]->(known:Student)-[:HATES]->(hated:Student)
RETURN s.name AS student, known.name AS known_person,
       hated.name AS hated_person
ORDER BY student, known_person, hated_person
""")
display(answer_1_7)
```

**Expected triples:** (Anna, Ben, Clara), (Eva, Frank, Ben).

</details>


## 1.8) Hating someone who is loved 
<a id="exercise-1-8"></a>

**Task:** Return each matching triple: a student who hates someone, that hated person, and a student who loves the hated person.


In [ ]:
# Exercise 1.8 -- add your code here


<details>
<summary>Show solution — Exercise 1.8</summary>

```python
answer_1_8 = run_cypher("""
MATCH (hater:Student)-[:HATES]->(person:Student)<-[:LOVES]-(lover:Student)
RETURN hater.name AS hater, person.name AS person,
       lover.name AS lover
ORDER BY hater, person, lover
""")
display(answer_1_8)
```

**Expected triples:** (Frank, Ben, Eva), (Frank, Ben, Grace), (Julia, Anna, Ben).

</details>


## 1.9) More than one distinct connection
<a id="exercise-1-9"></a>

**Task:** For each student, count **different people** reached through outgoing `KNOWS` or `BEST_FRIEND`. Return only students with more than one such person. Count people, not relationships; sort by name.


In [ ]:
# Exercise 1.9 -- add your code here


<details>
<summary>Show solution — Exercise 1.9</summary>

```python
answer_1_9 = run_cypher("""
MATCH (s:Student)-[:KNOWS|BEST_FRIEND]->(other:Student)
WITH s, count(DISTINCT other) AS connections
WHERE connections > 1
RETURN s.name AS student, connections
ORDER BY student
""")
display(answer_1_9)
```

**Expected:** Anna 2; Ben 2; Clara 3; David 2; Eva 4; Frank 2; Grace 2; Hannah 2; Julia 3. Ian has only one connection and is excluded. `WITH` makes the aggregated count available to the subsequent filter.

</details>


## 1.10) Best friends studying Data Science
<a id="exercise-1-10"></a>

**Task:** Return each student and the name of their best friend studying Data Science. Return distinct **student–friend pairs**, sorted by both names.


In [ ]:
# Exercise 1.10 -- add your code here


<details>
<summary>Show solution — Exercise 1.10</summary>

```python
answer_1_10 = run_cypher("""
MATCH (s:Student)-[:BEST_FRIEND]->(friend:Student)
WHERE friend.field_of_study = "Data Science"
RETURN DISTINCT s.name AS student, friend.name AS best_friend
ORDER BY student, best_friend
""")
display(answer_1_10)
```

**Expected pairs:** (Clara, Julia), (Eva, Anna), (Eva, Julia), (Hannah, Julia). These are four pairs involving three distinct students.

</details>


## 1.11) When the loving student's best friend hates the loved person 
<a id="exercise-1-11"></a>

**Task:** Find students who love someone hated by one of **the loving student's** best friends. Return the loving student, the loved person, and that best friend. The friendship belongs to the lover, not to the loved person.


In [ ]:
# Exercise 1.11 -- add your code here


<details>
<summary>Show solution — Exercise 1.11</summary>

```python
answer_1_11 = run_cypher("""
MATCH (lover:Student)-[:LOVES]->(loved:Student),
      (lover)-[:BEST_FRIEND]->(friend:Student),
      (friend)-[:HATES]->(loved)
RETURN lover.name AS lover, loved.name AS loved,
       friend.name AS hating_best_friend
ORDER BY lover, loved, hating_best_friend
""")
display(answer_1_11)
```

**Expected:** (Grace, Ben, Frank).

</details>


# Part 2: Tabular Data — Flight Knowledge Graph
<a id="flight-graph"></a>

Use only the supplied **small synthetic** CSV files in `data/`: 9 airports, 4 airlines, and 20 flight rows. 

### Graph model

| Node | Identity in this fixture | Properties |
|---|---|---|
| `Airport` | `airport_code` | `city`, `country` |
| `Airline` | `airline_code` | `name` |
| `Flight` | `flightNumber` | raw HHMM times and derived minute values |

```text
(Flight)-[:DEPARTS_FROM]->(Airport)
(Flight)-[:ARRIVES_AT]->(Airport)
(Flight)-[:OPERATED_BY]->(Airline)
```

The airline and airports are entities reached by relationships, rather than string references stored on a flight. A `Flight` node can itself connect to additional entities. In real operational data, a flight number alone does not identify a dated flight occurrence.

### Time interpretation and data quality

HHMM is a clock notation, not a duration: **07:45 − 06:30 = 75 minutes**, not the numeric subtraction `745 - 630`.

For this exercise only, interpret valid clocks in **one fictional common time zone on the same day**. The resulting `durationMinutes` is a teaching clock difference, not a claim about real travel time. Actual itineraries need dates and time-zone-aware timestamps.

The original CSVs are preserved:
- `DB006.arrivalTime = 5000` is invalid. Its `arrivalMinutes` and `durationMinutes` are absent/null.
- `LH011` has 13:10 → 12:20. Without a date or time zone, the duration is unknown. The importer **does not assume an overnight flight**.
- Both flight rows remain in the graph, with a `timeIssue` explanation. They are excluded only when a query needs a known duration.

Run the supplied preparation cells below. They validate columns, identifiers, and foreign keys before importing, and use parameterized `MERGE` instead of formatting CSV values into Cypher. The unchanged import can be rerun without adding duplicate nodes or relationships. No whole-database reset is performed.


In [ ]:
airports, airlines, flights, time_issues = get_flight_data(NOTEBOOK_DIR / "data")
print(f"CSV rows: {len(airports)} airports, {len(airlines)} airlines, {len(flights)} flights")
display(pd.DataFrame(time_issues))
display(pd.DataFrame(flights).head())


In [ ]:
# Return to the main instance if you previously explored Marvel.
print("Connected to:", connect())
flight_import = load_flights(NOTEBOOK_DIR / "data")
print(flight_import)
display(run_cypher("""
MATCH (f:Flight)-[r]->()
WHERE type(r) IN ["DEPARTS_FROM", "ARRIVES_AT", "OPERATED_BY"]
RETURN type(r) AS relationship, count(r) AS relationships
ORDER BY relationship
"""))


**Checkpoint:** the flight subgraph has **33 nodes and 60 relationships** (20 each of the three relationship types). The student graph can coexist in the same database because these exercises use different labels. Use `MUC` for Munich and `BER` for Berlin; airport codes are keys, while city spellings are display values.

The `departureTime` / `arrivalTime` properties keep the original HHMM integers. A returned value `630` means **06:30**. Derived `...Minutes` properties use minutes after midnight; missing or uncomputable values are `null` when queried.


## 2.1) Lufthansa from Munich to Berlin
<a id="exercise-2-1"></a>

**Task:** Find all Lufthansa (`airline_code = 'LH'`) flights from `MUC` to `BER`. Return the flight number and departure HHMM, ordered by departure time.


In [ ]:
# Exercise 2.1 -- add your code here


<details>
<summary>Show solution — Exercise 2.1</summary>

```python
answer_2_1 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(:Airport {airport_code: "MUC"}),
      (f)-[:ARRIVES_AT]->(:Airport {airport_code: "BER"}),
      (f)-[:OPERATED_BY]->(:Airline {airline_code: "LH"})
RETURN f.flightNumber AS flight, f.departureTime AS departure_hhmm
ORDER BY departure_hhmm, flight
""")
display(answer_2_1)
```

**Expected:** LH012 at 630 (06:30), LH013 at 1230 (12:30), LH014 at 2030 (20:30).

</details>


## 2.2) Destinations within two teaching hours
<a id="exercise-2-2"></a>

**Task:** Under the simplified time model above, return distinct destinations of Lufthansa flights departing `MUC` strictly after 06:00 with a known duration of at most 120 minutes. Do not replace duration by an arrival-before-08:00 test.


In [ ]:
# Exercise 2.2 -- add your code here


<details>
<summary>Show solution — Exercise 2.2</summary>

```python
answer_2_2 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(:Airport {airport_code: "MUC"}),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(:Airline {airline_code: "LH"})
WHERE f.departureMinutes > 360
  AND f.durationMinutes IS NOT NULL
  AND f.durationMinutes <= 120
RETURN DISTINCT d.airport_code AS destination, d.city AS city
ORDER BY destination
""")
display(answer_2_2)
```

**Expected:** BER (Berlin), HAM (Hamburg), LHR (London). Five flight rows reach these three destinations. This uses the stated fictional common time zone, not real elapsed travel time to London.

</details>


## 2.3) All Lufthansa departures from Munich 
<a id="exercise-2-3"></a>

**Task:** List Lufthansa flights departing `MUC`. Include flight number, destination airport, departure HHMM, and arrival HHMM. Sort by flight number.


In [ ]:
# Exercise 2.3 -- add your code here


<details>
<summary>Show solution — Exercise 2.3</summary>

```python
answer_2_3 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(:Airport {airport_code: "MUC"}),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(:Airline {airline_code: "LH"})
RETURN f.flightNumber AS flight, d.airport_code AS destination,
       f.departureTime AS departure_hhmm, f.arrivalTime AS arrival_hhmm
ORDER BY flight
""")
display(answer_2_3)
```

**Expected flight numbers:** LH012, LH013, LH014, LH016, LH018.

</details>


## 2.4) Flights arriving in Berlin 
<a id="exercise-2-4"></a>

**Task:** List all flights arriving at `BER`, including flight number, origin airport, airline, raw arrival HHMM, and any time-quality issue. Retain a flight even when its arrival clock is invalid.


In [ ]:
# Exercise 2.4 -- add your code here


<details>
<summary>Show solution — Exercise 2.4</summary>

```python
answer_2_4 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(:Airport {airport_code: "BER"}),
      (f)-[:OPERATED_BY]->(a:Airline)
RETURN f.flightNumber AS flight, o.airport_code AS origin,
       a.name AS airline, f.arrivalTime AS arrival_hhmm,
       f.timeIssue AS time_issue
ORDER BY flight
""")
display(answer_2_4)
```

**Expected:** DB006, LH009, LH012, LH013, LH014, LH019. DB006 still reports the original invalid raw value 5000 and an issue message; it is not silently corrected.

</details>


## 2.5) All Lufthansa flights 
<a id="exercise-2-5"></a>

**Task:** List all Lufthansa flight numbers with origin, destination, departure HHMM, and arrival HHMM. Sort by flight number.


In [ ]:
# Exercise 2.5 -- add your code here


<details>
<summary>Show solution — Exercise 2.5</summary>

```python
answer_2_5 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(:Airline {airline_code: "LH"})
RETURN f.flightNumber AS flight, o.airport_code AS origin,
       d.airport_code AS destination,
       f.departureTime AS departure_hhmm, f.arrivalTime AS arrival_hhmm
ORDER BY flight
""")
display(answer_2_5)
```

**Expected:** 13 flight rows. LH011 remains listed even though its duration is unknown.

</details>


## 2.6) All flight rows and operating airlines 
<a id="exercise-2-6"></a>

**Task:** List every flight with origin, destination, airline, and both raw HHMM values. Return flight numbers so individual flights remain distinguishable.


In [ ]:
# Exercise 2.6 -- add your code here


<details>
<summary>Show solution — Exercise 2.6</summary>

```python
answer_2_6 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(a:Airline)
RETURN f.flightNumber AS flight, o.airport_code AS origin,
       d.airport_code AS destination, a.name AS airline,
       f.departureTime AS departure_hhmm, f.arrivalTime AS arrival_hhmm
ORDER BY flight
""")
display(answer_2_6)
```

**Expected:** 20 rows, one per flight in the unchanged fixture.

</details>


## 2.7) Distinct direct airport connections 
<a id="exercise-2-7"></a>

**Task:** Each CSV row represents one nonstop flight leg. Return distinct origin–destination airport pairs, ignoring flight number and airline. Explain why there are fewer pairs than flight rows.


In [ ]:
# Exercise 2.7 -- add your code here


<details>
<summary>Show solution — Exercise 2.7</summary>

```python
answer_2_7 = run_cypher("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport)
RETURN DISTINCT o.airport_code AS origin, d.airport_code AS destination
ORDER BY origin, destination
""")
display(answer_2_7)
```

**Expected:** 13 pairs from 20 flight rows. Several flights share STR → FRA or MUC → BER. Projection plus `DISTINCT` removes repeated endpoint pairs.

</details>


## 2.8) Airports receiving at least two airlines
<a id="exercise-2-8"></a>

**Task:** Find airports with arriving flights operated by at least two distinct airlines. Group by the airport node, not merely its city name. Return airport code, city, and airline count.


In [ ]:
# Exercise 2.8 -- add your code here


<details>
<summary>Show solution — Exercise 2.8</summary>

```python
answer_2_8 = run_cypher("""
MATCH (f:Flight)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(a:Airline)
WITH d, count(DISTINCT a) AS airlines
WHERE airlines >= 2
RETURN d.airport_code AS airport, d.city AS city, airlines
ORDER BY airport
""")
display(answer_2_8)
```

**Expected:** BER, FRA, JFK, LHR; each has two distinct arriving airlines.

</details>


## 2.9) Bounded routes with stopovers 
<a id="exercise-2-9"></a>

**Task:** Enumerate topological routes with zero, one, or two stopovers, represented by one, two, or three flight nodes. Return origin, ordered flight-number list, destination, and number of stopovers. Ignore schedules and time-quality issues in this task; these are graph connections, not bookable itineraries. **Hint:** a route alternates `Airport ← Flight → Airport`. Repeating only `DEPARTS_FROM` cannot express it. Use three bounded patterns joined with `UNION ALL`, not an unbounded expansion.


In [ ]:
# Exercise 2.9 -- add your code here


<details>
<summary>Show solution — Exercise 2.9</summary>

```python
answer_2_9 = run_cypher("""
MATCH (o:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(d:Airport)
RETURN o.airport_code AS origin, [f.flightNumber] AS flights,
       d.airport_code AS destination, 0 AS stopovers
UNION ALL
MATCH (o:Airport)<-[:DEPARTS_FROM]-(f1:Flight)-[:ARRIVES_AT]->(via:Airport)
      <-[:DEPARTS_FROM]-(f2:Flight)-[:ARRIVES_AT]->(d:Airport)
WHERE f1 <> f2
RETURN o.airport_code AS origin,
       [f1.flightNumber, f2.flightNumber] AS flights,
       d.airport_code AS destination, 1 AS stopovers
UNION ALL
MATCH (o:Airport)<-[:DEPARTS_FROM]-(f1:Flight)-[:ARRIVES_AT]->(v1:Airport)
      <-[:DEPARTS_FROM]-(f2:Flight)-[:ARRIVES_AT]->(v2:Airport)
      <-[:DEPARTS_FROM]-(f3:Flight)-[:ARRIVES_AT]->(d:Airport)
WHERE f1 <> f2 AND f1 <> f3 AND f2 <> f3
RETURN o.airport_code AS origin,
       [f1.flightNumber, f2.flightNumber, f3.flightNumber] AS flights,
       d.airport_code AS destination, 2 AS stopovers
""")
display(answer_2_9)
```

**Expected:** 45 flight sequences: 20 direct, 21 with one stopover, and 4 with two. Result order is unspecified. Valid schedules additionally need comparable timestamps, transfer buffers, and dated flight-occurrence identifiers.

</details>


## 2.10) Lufthansa before noon 
<a id="exercise-2-10"></a>

**Task:** Find Lufthansa flights departing strictly before 12:00 noon. Filter using minutes after midnight and return flight number, origin, destination, and departure HHMM.


In [ ]:
# Exercise 2.10 -- add your code here


<details>
<summary>Show solution — Exercise 2.10</summary>

```python
answer_2_10 = run_cypher("""
MATCH (f:Flight)-[:OPERATED_BY]->(:Airline {airline_code: "LH"}),
      (f)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport)
WHERE f.departureMinutes < 720
RETURN f.flightNumber AS flight, o.airport_code AS origin,
       d.airport_code AS destination, f.departureTime AS departure_hhmm
ORDER BY flight
""")
display(answer_2_10)
```

**Expected:** LH001, LH004, LH005, LH007, LH008, LH009, LH012.

</details>


## 2.11) Incoming flights, including airports with none
<a id="exercise-2-11"></a>

**Task:** Rank all airports by the number of distinct incoming flights. Include airports with zero incoming flights. Sort by count descending, then airport code ascending. Which airport ranks first, and why does `count(*)` give a wrong zero-case count?


In [ ]:
# Exercise 2.11 -- add your code here


<details>
<summary>Show solution — Exercise 2.11</summary>

```python
answer_2_11 = run_cypher("""
MATCH (a:Airport)
OPTIONAL MATCH (f:Flight)-[:ARRIVES_AT]->(a)
RETURN a.airport_code AS airport, count(DISTINCT f) AS incoming_flights
ORDER BY incoming_flights DESC, airport
""")
display(answer_2_11)
```

**Expected:** BER 6; FRA 4; JFK 4; DUS 2; LHR 2; CDG 1; HAM 1; MUC 0; STR 0. `OPTIONAL MATCH` retains an airport with no matching flight; `count(f)` ignores the resulting null, whereas `count(*)` counts that retained row.

</details>


# Part 3: Querying the Marvel Comic Dataset
<a id="marvel-graph"></a>

### Prepare a separate instance before class

The official sample-dataset importer expects an **empty database** and loading a dataset twice creates duplicates. Keep the student/flight instance intact and use a second Compose project:

```bash
docker compose --env-file .env.marvel -p tut03-marvel up -d --wait
docker compose --env-file .env.marvel -p tut03-marvel ps
```

Open **[http://localhost:3001](http://localhost:3001)**, the Lab for this second stack. Inside that Lab, the database service is still `memgraph:7687`. From local Python, this stack is reached at **`bolt://localhost:7690`**.

In Lab, open **Datasets**, choose the catalog entry **Marvel Cinematic Universe** (the supplied graph describes comic appearances), and load it **once** into this fresh instance. Wait for import completion. Do not load it again when resuming a session; the named volume retains the data.

The [official dataset catalog](https://memgraph.com/docs/querying/sample-datasets) documents this import and links the source. The catalog is external and can change; run the checks below rather than assuming fixed counts. No synthetic replacement dataset is silently substituted.

### Data model

| Label | Relevant properties |
|---|---|
| `Hero` | `name`, often including both alias and identity |
| `Comic` | `name`, including issue/volume when available |
| `ComicSeries` | `title`, `publishYear` as a list of recorded series publication years |

```text
(Hero)-[:AppearedIn]->(Comic)-[:IsPartOfSeries]->(ComicSeries)
(Hero)-[:AppearedInSameComic]->(Hero)
```

Name-substring tasks deliberately describe a **lexical selection**, which can match multiple characters or issues. They do not establish canonical identity. When a single character is intended, use its exact stored name.

A series having a recorded year after 2000 does **not** establish that a particular comic issue was published after 2000.


In [ ]:
marvel_uri = os.environ.get("MARVEL_URI", "bolt://localhost:7690")
print("Connected to Marvel:", connect(uri=marvel_uri))
marvel_check = run_cypher("""
MATCH (h:Hero) RETURN "Hero" AS label, count(h) AS nodes
UNION ALL
MATCH (c:Comic) RETURN "Comic" AS label, count(c) AS nodes
UNION ALL
MATCH (s:ComicSeries) RETURN "ComicSeries" AS label, count(s) AS nodes
""")
display(marvel_check)
if not all(int(count) > 0 for count in marvel_check["nodes"]):
    raise RuntimeError(
        "Load the official Marvel dataset once in Lab on localhost:3001, "
        "wait for completion, and rerun this preparation cell."
    )
display(run_cypher("""
MATCH (s:ComicSeries)
WHERE s.publishYear IS NOT NULL
RETURN s.title AS series, s.publishYear AS recorded_years
ORDER BY series LIMIT 5
"""))


**Query notes:** relationship types are case-sensitive: use `AppearedIn` and `IsPartOfSeries` exactly. Use `ANY(year IN coalesce(series.publishYear, []) WHERE toInteger(year) > 1960)` for the list of series years. `coalesce` supplies an empty list when the property is missing.

The reference answers use the common `AppearedIn` path to compute co-appearances, so a precomputed hero–hero edge is not required for task 3.4. Dataset-dependent result lists are inspected on the loaded catalog version; the model notebook contains no stale saved outputs.


## 3.1) Count the three kinds of entity
<a id="exercise-3-1"></a>

**Task:** Return one row each for `Hero`, `Comic`, and `ComicSeries`, containing the label and node count. This is a count query, not a full listing of every hero.


In [ ]:
# Exercise 3.1 -- add your code here


<details>
<summary>Show solution — Exercise 3.1</summary>

```python
answer_3_1 = run_cypher("""
MATCH (h:Hero) RETURN "Hero" AS label, count(h) AS nodes
UNION ALL
MATCH (c:Comic) RETURN "Comic" AS label, count(c) AS nodes
UNION ALL
MATCH (s:ComicSeries) RETURN "ComicSeries" AS label, count(s) AS nodes
""")
display(answer_3_1)
```

**Check:** three rows with positive counts after a completed import. Order across the `UNION ALL` branches is not guaranteed. Record the counts of the version you loaded.

</details>


## 3.2) Ten hero names 
<a id="exercise-3-2"></a>

**Task:** Return ten distinct non-empty hero names in alphabetical order. Exclude missing names and names containing only whitespace. The ordering makes the selected ten reproducible for an unchanged dataset.


In [ ]:
# Exercise 3.2 -- add your code here


<details>
<summary>Show solution — Exercise 3.2</summary>

```python
answer_3_2 = run_cypher("""
MATCH (h:Hero)
WHERE h.name IS NOT NULL AND trim(h.name) <> ""
RETURN DISTINCT h.name AS hero
ORDER BY hero
LIMIT 10
""")
display(answer_3_2)
```

**Check:** at most ten unique non-empty names, ordered alphabetically. Missing and whitespace-only names are excluded.

</details>


## 3.3) Spider-Man / Peter Parker's comic appearances 
<a id="exercise-3-3"></a>

**Task:** Use the exact stored name `SPIDER-MAN/PETER PARKER`. Return distinct comic names where that hero appears, ordered by comic name. Explain why a substring search could select additional identities.


In [ ]:
# Exercise 3.3 -- add your code here


<details>
<summary>Show solution — Exercise 3.3</summary>

```python
answer_3_3 = run_cypher("""
MATCH (:Hero {name: "SPIDER-MAN/PETER PARKER"})-[:AppearedIn]->(c:Comic)
RETURN DISTINCT c.name AS comic
ORDER BY comic
""")
display(answer_3_3)
```

**Check:** only appearances of that exact stored Hero node are selected. If empty, first inspect the actual names in the loaded dataset instead of broadening the identity silently.

</details>


## 3.4) Co-appearing with an Iron Man name match
<a id="exercise-3-4"></a>

**Task:** Find ten distinct other hero names appearing in a shared comic with any hero whose name contains `iron man`, ignoring case. Exclude the matched Iron Man node itself and sort by the returned name. Derive co-appearance through shared `Comic` nodes.


In [ ]:
# Exercise 3.4 -- add your code here


<details>
<summary>Show solution — Exercise 3.4</summary>

```python
answer_3_4 = run_cypher("""
MATCH (iron:Hero)-[:AppearedIn]->(c:Comic)<-[:AppearedIn]-(other:Hero)
WHERE toLower(iron.name) CONTAINS "iron man" AND other <> iron
RETURN DISTINCT other.name AS hero
ORDER BY hero
LIMIT 10
""")
display(answer_3_4)
```

**Check:** at most ten distinct names. Many comic witnesses can yield the same hero; projection plus `DISTINCT` removes those repeated result names.

</details>


## 3.5) Thor name matches, comics, and series 
<a id="exercise-3-5"></a>

**Task:** For each Hero name containing `thor` (case-insensitive), return the hero name, comic name, and series title reached through `AppearedIn` and `IsPartOfSeries`. This is explicitly a name-substring selection, not a proof of character identity.


In [ ]:
# Exercise 3.5 -- add your code here


<details>
<summary>Show solution — Exercise 3.5</summary>

```python
answer_3_5 = run_cypher("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(s:ComicSeries)
WHERE toLower(h.name) CONTAINS "thor"
RETURN DISTINCT h.name AS hero, c.name AS comic, s.title AS series
ORDER BY hero, comic, series
""")
display(answer_3_5)
```

**Check:** each row includes a complete hero → comic → series path. Different name variants can appear.

</details>


## 3.6) Series with a recorded year after 1960 
<a id="exercise-3-6"></a>

**Task:** Find five series with at least one recorded publication year strictly greater than 1960. Sort by title. Do not compare a list directly with a scalar year.


In [ ]:
# Exercise 3.6 -- add your code here


<details>
<summary>Show solution — Exercise 3.6</summary>

```python
answer_3_6 = run_cypher("""
MATCH (s:ComicSeries)
WHERE ANY(year IN coalesce(s.publishYear, []) WHERE toInteger(year) > 1960)
RETURN s.title AS series, s.publishYear AS recorded_years
ORDER BY series
LIMIT 5
""")
display(answer_3_6)
```

**Check:** each returned list has a year after 1960. This does not mean that the first publication of the series was after 1960.

</details>


## 3.7) Heroes linked to a series with a year after 2000
<a id="exercise-3-7"></a>

**Task:** Return twenty distinct hero names appearing in comics belonging to a series with at least one recorded publication year strictly after 2000. Sort by hero name. State why this does not date the individual comic appearance.


In [ ]:
# Exercise 3.7 -- add your code here


<details>
<summary>Show solution — Exercise 3.7</summary>

```python
answer_3_7 = run_cypher("""
MATCH (h:Hero)-[:AppearedIn]->(:Comic)-[:IsPartOfSeries]->(s:ComicSeries)
WHERE ANY(year IN coalesce(s.publishYear, []) WHERE toInteger(year) > 2000)
RETURN DISTINCT h.name AS hero
ORDER BY hero
LIMIT 20
""")
display(answer_3_7)
```

**Check:** at most twenty unique names. The date condition applies to the series, not to a recorded publication date of the specific issue. This task occurs once; the former repeated copies have been removed.

</details>


## 3.8) More than one hundred different comics 
<a id="exercise-3-8"></a>

**Task:** Return heroes who appear in more than 100 distinct Comic nodes. Return the hero's name and the count, ordered by count descending and then name.


In [ ]:
# Exercise 3.8 -- add your code here


<details>
<summary>Show solution — Exercise 3.8</summary>

```python
answer_3_8 = run_cypher("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)
WITH h, count(DISTINCT c) AS appearances
WHERE appearances > 100
RETURN h.name AS hero, appearances
ORDER BY appearances DESC, hero
""")
display(answer_3_8)
```

**Check:** every count is strictly greater than 100. Counting distinct comics avoids counting duplicate appearance relationships as additional comics.

</details>


## 3.9) Seven distinct heroes in Captain Marvel name matches 
<a id="exercise-3-9"></a>

**Task:** Return seven distinct hero names with an appearance in a comic whose name contains `captain marvel`, ignoring case. This can select multiple issues. Sort names before limiting; do not limit hero–comic pairs when the task asks for heroes.


In [ ]:
# Exercise 3.9 -- add your code here


<details>
<summary>Show solution — Exercise 3.9</summary>

```python
answer_3_9 = run_cypher("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)
WHERE toLower(c.name) CONTAINS "captain marvel"
RETURN DISTINCT h.name AS hero
ORDER BY hero
LIMIT 7
""")
display(answer_3_9)
```

**Check:** at most seven distinct names. Filtering only on `marvel` would be broader than the question.

</details>


## 3.10) Captain Marvel issues and their series 
<a id="exercise-3-10"></a>

**Task:** Return each matching comic name and its series title for comic names containing `captain marvel`, ignoring case. Do not assume the phrase identifies one unique issue.


In [ ]:
# Exercise 3.10 -- add your code here


<details>
<summary>Show solution — Exercise 3.10</summary>

```python
answer_3_10 = run_cypher("""
MATCH (c:Comic)-[:IsPartOfSeries]->(s:ComicSeries)
WHERE toLower(c.name) CONTAINS "captain marvel"
RETURN DISTINCT c.name AS comic, s.title AS series
ORDER BY comic, series
""")
display(answer_3_10)
```

**Check:** each returned pair is supported by an `IsPartOfSeries` relationship.

</details>


In [ ]:
close()
